In [1]:
import sympy as sp

from lib.sp_model import (
    FR,
    Q1,
    Q2,
    Q3,
    T0,
    T1,
    T2,
    T3,
    Ff1,
    Ff2,
    dT1dt,
    dT2dt,
    dT3dt,
    dxA1dt,
    dxA2dt,
    dxA3dt,
    dxB1dt,
    dxB2dt,
    dxB3dt,
    xA1,
    xA2,
    xA3,
    xB1,
    xB2,
    xB3,
)

f = sp.Matrix([dT1dt, dT2dt, dT3dt, dxA1dt, dxB1dt, dxA2dt, dxB2dt, dxA3dt, dxB3dt])
X = sp.Matrix([T1, T2, T3, xA1, xB1, xA2, xB2, xA3, xB3])
U = sp.Matrix([Ff1, Ff2, FR, Q1, Q2, Q3, T0])


# Linearization


In [2]:
from lib.parameters import (
    FR_0,
    Q1_0,
    Q2_0,
    Q3_0,
    T0_0,
    T1_0,
    T2_0,
    T3_0,
    Ff1_0,
    Ff2_0,
    xA1_0,
    xA2_0,
    xA3_0,
    xB1_0,
    xB2_0,
    xB3_0,
)

U0_vals = {Ff1: Ff1_0, Ff2: Ff2_0, FR: FR_0, Q1: Q1_0, Q2: Q2_0, Q3: Q3_0, T0: T0_0}

X0_vals = {
    T1: T1_0,
    T2: T2_0,
    T3: T3_0,
    xA1: xA1_0,
    xB1: xB1_0,
    xA2: xA2_0,
    xB2: xB2_0,
    xA3: xA3_0,
    xB3: xB3_0,
}

A = f.jacobian(X).subs(X0_vals).subs(U0_vals)
B = f.jacobian(U).subs(X0_vals).subs(U0_vals)

f_linear = A @ X + B @ U


# Transfer Functions


In [3]:
import hickle as hkl

s = sp.symbols("s")
I = sp.eye(A.rows)
G = (s * I - A).inv() * B

# Simplify the transfer matrix
G = sp.simplify(G)

with open("../outputs/funções_de_transferência.txt", "w", encoding="utf-8") as f:
    for i in range(G.rows):
        for j in range(G.cols):
            f.write(f"G[{i},{j}] =\n")
            f.write(str(G[i, j]))
            f.write("\n\n")

with open("../outputs/funções_de_transferência.tex", "w", encoding="utf-8") as f:
    for i in range(G.rows):
        for j in range(G.cols):
            expr = G[i, j].evalf(3)  # Evaluate the expression to 3 significant digits
            f.write(f"G_{{{i + 1},{j + 1}}} = ")
            f.write(sp.latex(expr))
            f.write("\n\n")

hkl.dump(G, "../outputs/G.hkl")


/home/silas/workspace/ENGF93/.venv/lib/python3.14/site-packages/hickle/lookup.py:1491: SerializedWarning: 'ImmutableDenseMatrix' type not understood, data is serialized:
  warnings.warn(


# Coupled state equations (for scilab)


In [4]:
from lib.utils import input_labels, output_labels

labels = output_labels + input_labels

latex_labels = {sym: label.strip("$") for sym, label in zip(list(X) + list(U), labels)}


all_vars = X.col_join(U)
theta_dict = {}

linhas_scilab = []
f_linear_theta = []

for i, eq in enumerate(f_linear):
    estado = str(X[i])
    nova_eq = 0

    for var in all_vars:
        coef = eq.coeff(var)

        if coef != 0:
            var_name = str(var)
            estado = latex_labels[X[i]]
            var_name = latex_labels[var]
            theta_name = rf"\theta_{{{estado},{var_name}}}"
            theta = sp.Symbol(theta_name)

            theta_dict[coef] = theta
            nova_eq += theta * var

            coef_val = float(coef)
            coef_str = f"{coef_val:.4g}".replace(".", "{,}")
            linhas_scilab.append(f"{theta_name} &= {coef_str}")

    f_linear_theta.append(nova_eq)

f_linear_theta = sp.Matrix(f_linear_theta)

agrupadas = []
for i in range(0, len(linhas_scilab), 5):
    agrupadas.append(" & ".join(linhas_scilab[i : i + 5]) + r" \\")

with open("../outputs/theta_values.txt", "w", encoding="utf-8") as f:
    for linha in agrupadas:
        f.write(linha + "\n")


In [5]:
Xs = sp.Matrix([sp.Symbol(f"{x}_s") for x in X])
Us = sp.Matrix([sp.Symbol(f"{u}_s") for u in U])

subs_laplace = {}

for i in range(len(X)):
    subs_laplace[X[i]] = Xs[i]

for i in range(len(U)):
    subs_laplace[U[i]] = Us[i]

eqs_s = []

for i in range(len(X)):
    rhs = f_linear_theta[i].subs(subs_laplace)  # type: ignore

    eq = sp.solve(sp.Eq(s * Xs[i], rhs), Xs[i])[0]

    eqs_s.append(sp.simplify(eq))


with open("../outputs/laplace.txt", "w", encoding="utf-8") as f:
    for i, eq in enumerate(eqs_s):
        f.write(f"{X[i]}(s) =\n")
        f.write(str(eq))
        f.write("\n\n")
